# 05 — Phân tích kết quả

**Đầu vào:** `GraphK/runs/` (notebook 04), `GraphK/processed/junyi/` (notebook 02, 03).
**Đầu ra:** `GraphK/analysis/` gồm các bảng (`.csv`), hình (`.png`) và `tom_tat.json` (số liệu cho báo cáo).

Các phân tích:
1. **Bảng 1** — AUC, ACC, RMSE trên tập kiểm tra của mọi thực nghiệm (RQ1, RQ2).
2. **Bảng 2** — nghiên cứu cắt bỏ: xử lý nút cô lập, lan truyền một chiều/hai chiều.
3. **Bảng 3** — AUC trên tập khái niệm giữ lại và trên các khái niệm còn lại (RQ3).
4. **Bảng 4** — AUC theo nhóm khái niệm: cô lập, bậc thấp, bậc cao trong đồ thị chuyên gia; khái niệm cô lập trong đồ thị thống kê.
5. **Kiểm định DeLong ghép cặp** cho hiệu AUC giữa các mô hình, trên toàn tập kiểm tra và trên khái niệm giữ lại. Kiểm định giả định các dự đoán độc lập (bỏ qua tương quan trong cùng người học), và với hàng triệu dự đoán thì hiệu rất nhỏ cũng có ý nghĩa thống kê. Vì vậy cần đọc kèm độ lớn hiệu AUC.
6. **Hình** (theo kế hoạch hình của dàn ý): Hình 2 so sánh cấu trúc đồ thị, Hình 3 đường cong học, Hình 4 AUC trên khái niệm giữ lại, Hình 5 case study. Hình 1 (sơ đồ kiến trúc) được vẽ bằng `scripts/ve_so_do_kien_truc.py`.
7. **Case study** — một người học trong tập kiểm tra làm bài trên một chuỗi tiên quyết; mức thành thạo dự đoán theo từng bước của GKT-Expert, GKT-Stat và DKT.

Không cần GPU.

In [ ]:
# Gắn Google Drive, lấy mã nguồn mới nhất của kho GraphK và cài gói graphk
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DRIVE = '/content/drive/MyDrive/GraphK'
REPO = '/content/GraphK'
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists(REPO):
    !git clone -q https://github.com/Trilosphan/GraphK.git {REPO}
else:
    !git -C {REPO} pull -q
%pip install -q -e {REPO}
sys.path.insert(0, f'{REPO}/src')
!git -C {REPO} log --oneline -1

## 0. Cấu hình

In [ ]:
import json
from pathlib import Path

PROC = Path(DRIVE) / 'processed' / 'junyi'
RUNS = Path(DRIVE) / 'runs'
OUT = Path(DRIVE) / 'analysis'
OUT.mkdir(parents=True, exist_ok=True)
FOLD, SEED = 0, 42

name = lambda model, graph, tag='': f'{model}-{graph}-fold{FOLD}-seed{SEED}' + (f'-{tag}' if tag else '')
MAIN = {'GKT-Expert': name('gkt', 'expert'), 'GKT-Stat': name('gkt', 'stat'), 'GKT-Random': name('gkt', 'random'),
        'DKT': name('dkt', 'none'), 'AKT': name('akt', 'none')}
ABL = {'GKT-Expert (hai chiều, tự cập nhật)': name('gkt', 'expert'),
       'GKT-Expert + nút gốc ảo': name('gkt', 'expert', 'root'),
       'GKT-Expert một chiều': name('gkt', 'expert', 'unidir')}

## 1. Bảng 1 — Kết quả trên tập kiểm tra

In [ ]:
import pandas as pd
from graphk.analysis import load_results, results_table

results = load_results(RUNS)
table = results_table(results)
label = {v: k for k, v in {**ABL, **MAIN}.items()}
table.insert(0, 'mô hình', table['thực nghiệm'].map(label))
table = table.dropna(subset=['mô hình']).sort_values('AUC kiểm tra', ascending=False)
table.to_csv(OUT / 'bang1_ket_qua.csv', index=False)
print('Thực nghiệm chưa có kết quả:', [k for k, v in MAIN.items() if v not in results] or 'không có')
table.round(4)

## 2. Bảng 2 — Nghiên cứu cắt bỏ

In [ ]:
abl = table[table['thực nghiệm'].isin(ABL.values())].copy()
abl['mô hình'] = abl['thực nghiệm'].map({v: k for k, v in ABL.items()})
abl.to_csv(OUT / 'bang2_cat_bo.csv', index=False)
abl[['mô hình', 'epoch', 'AUC kiểm định', 'AUC kiểm tra', 'AUC khái niệm giữ lại']].round(4)

## 3. Bảng 3 — Khái niệm giữ lại (RQ3)

In [ ]:
main = table[table['thực nghiệm'].isin(MAIN.values())].copy()
main['mô hình'] = main['thực nghiệm'].map({v: k for k, v in MAIN.items()})
b3 = main[['mô hình', 'AUC kiểm tra', 'AUC khái niệm còn lại', 'AUC khái niệm giữ lại']].copy()
b3['chênh lệch (còn lại − giữ lại)'] = b3['AUC khái niệm còn lại'] - b3['AUC khái niệm giữ lại']
b3.to_csv(OUT / 'bang3_khai_niem_giu_lai.csv', index=False)
b3.round(4)

## 4. Bảng 4 — AUC theo nhóm khái niệm

In [ ]:
import numpy as np
import scipy.sparse as sp
from graphk.analysis import load_predictions, node_groups, auc_by_group

preds = {k: load_predictions(RUNS, v) for k, v in MAIN.items() if v in results}
groups = node_groups(sp.load_npz(PROC / 'adj_expert.npz'), sp.load_npz(PROC / 'graphs' / f'adj_stat_fold{FOLD}.npz'))
b4a = auc_by_group(preds, groups['nhóm chuyên gia'].to_numpy())
stat_iso = np.where(groups['cô lập trong đồ thị thống kê'], 'cô lập trong đồ thị thống kê', 'có cạnh trong đồ thị thống kê')
b4b = auc_by_group(preds, stat_iso)
b4 = pd.concat([b4a, b4b])
counts = {g: int(np.isin(next(iter(preds.values()))['q'], groups.index[(groups['nhóm chuyên gia'] == g)]).sum())
          for g in b4a.index}
b4.to_csv(OUT / 'bang4_nhom_khai_niem.csv')
print('Số dự đoán theo nhóm chuyên gia:', counts)
b4.round(4)

## 5. Kiểm định DeLong ghép cặp

In [ ]:
from graphk.analysis import pairwise_tests

heldout = pd.read_csv(PROC / 'graphs' / 'heldout_concepts.csv')['node_id'].to_numpy()
pairs = [('GKT-Expert', 'GKT-Stat'), ('GKT-Expert', 'GKT-Random'), ('GKT-Expert', 'DKT'),
         ('GKT-Expert', 'AKT'), ('GKT-Stat', 'GKT-Random'), ('DKT', 'AKT')]
t_all = pairwise_tests(preds, pairs).assign(tập='toàn bộ')
t_held = pairwise_tests(preds, pairs, mask_fn=lambda q: np.isin(q, heldout)).assign(tập='khái niệm giữ lại')
tests = pd.concat([t_all, t_held], ignore_index=True)
tests.to_csv(OUT / 'kiem_dinh_delong.csv', index=False)
tests[['tập', 'mô hình 1', 'mô hình 2', 'số dự đoán', 'auc_1', 'auc_2', 'delta', 'se', 'p_value']].round(5)

## 6. Hình

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for k, v in MAIN.items():
    if v in results:
        h = results[v]['history']
        ax.plot([e['epoch'] for e in h], [e['val']['all']['auc'] for e in h], marker='o', label=k)
ax.set_xlabel('epoch'); ax.set_ylabel('AUC kiểm định'); ax.set_title('Đường cong học (phần 0)')
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(OUT / 'hinh3_duong_cong_hoc.png', dpi=150)

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(b3))
ax.bar(x - 0.2, b3['AUC kiểm tra'], 0.4, label='toàn tập kiểm tra')
ax.bar(x + 0.2, b3['AUC khái niệm giữ lại'], 0.4, label='khái niệm giữ lại')
ax.set_xticks(x, b3['mô hình'], rotation=15)
lo = min(b3['AUC kiểm tra'].min(), b3['AUC khái niệm giữ lại'].min())
ax.set_ylim(lo - 0.02, None); ax.set_ylabel('AUC'); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_title('AUC trên toàn tập và trên khái niệm giữ lại')
fig.tight_layout(); fig.savefig(OUT / 'hinh4_auc_giu_lai.png', dpi=150)
plt.show()

## 7. Hình 2 — Đồ thị chuyên gia, thống kê và ngẫu nhiên

In [ ]:
from graphk.graphs import load_processed, fold_users, interactions_of_users, mastery_matrices, added_value_scores, edge_set
from graphk.analysis import plot_graph_comparison

data = load_processed(PROC)
adj_e = data['adj_expert']
adj_s = sp.load_npz(PROC / 'graphs' / f'adj_stat_fold{FOLD}.npz')
adj_r = sp.load_npz(PROC / 'graphs' / f'adj_random_fold{FOLD}.npz')
u, qq, cc = interactions_of_users(data['sequences'], fold_users(data['splits'], FOLD))
att, mas = mastery_matrices(u, qq, cc, len(data['nodes']), 0.5)
av, _ = added_value_scores(att, mas, 50)
ex = list(edge_set(adj_e))
av_groups = {'chuyên gia (xuôi)': np.array([av[a, b] for a, b in ex]),
             'chuyên gia (ngược)': np.array([av[b, a] for a, b in ex]),
             'cạnh thống kê': np.array([av[a, b] for a, b in edge_set(adj_s)]),
             'mọi cặp đủ hỗ trợ': av[np.isfinite(av)]}
plot_graph_comparison({'chuyên gia': adj_e, 'thống kê': adj_s, 'ngẫu nhiên': adj_r}, av_groups, OUT / 'hinh2_do_thi.png')
plt.show()

## 8. Case study — Hình 5

Chọn tất định một đoạn chuỗi dài 25–60 bước trong tập kiểm tra có nhiều cạnh tiên quyết chuyên gia nhất (khái niệm tiên quyết xuất hiện trước khái niệm phụ thuộc, và kết quả trên khái niệm phụ thuộc có cả đúng lẫn sai). Với mỗi mô hình, `trace` cho xác suất trả lời đúng mọi khái niệm sau từng bước.

In [ ]:
from graphk.analysis import select_case_study, load_trained, case_study_traces, plot_case_study
from graphk.train import segment_indices

test_users = data['splits'].loc[data['splits'].split == 'test', 'user_id']
seg, chain = select_case_study(data['sequences'], segment_indices(data['sequences'], test_users), adj_e)
focus = sorted({c for e in chain for c in e})[:6]
names = data['nodes'].set_index('node_id').loc[focus, 'exercise_name'].tolist()
models = {k: load_trained(RUNS, MAIN[k], PROC, data) for k in ('GKT-Expert', 'GKT-Stat', 'DKT') if MAIN[k] in results}
case = case_study_traces(models, data['sequences'], seg, focus)
case.update({'segment': int(seg), 'user_id': int(data['sequences']['seq_user'][seg]), 'chain': chain,
             'focus_names': names,
             'q_names': data['nodes'].set_index('node_id').loc[case['q'], 'exercise_name'].tolist()})
y_next = np.array(case['r'][1:])
case['accuracy_next'] = {k: float(((np.array(v) >= 0.5) == y_next).mean()) for k, v in case['next_pred'].items()}
print('Đoạn', seg, '| người học', case['user_id'], '| độ dài', len(case['q']))
print('Cạnh tiên quyết trong đoạn:', [(names[focus.index(a)] if a in focus else a, names[focus.index(b)] if b in focus else b) for a, b in chain])
print('Độ chính xác dự đoán bước kế tiếp:', case['accuracy_next'])
plot_case_study(case, names, OUT / 'hinh5_case_study.png')
(OUT / 'case_study.json').write_text(json.dumps(case, ensure_ascii=False, indent=1))
plt.show()

## 9. Tóm tắt cho báo cáo

In [ ]:
import json

summary = {
    'bang1': table.round(5).to_dict(orient='records'),
    'bang2': abl.round(5).to_dict(orient='records'),
    'bang3': b3.round(5).to_dict(orient='records'),
    'bang4': b4.round(5).reset_index().to_dict(orient='records'),
    'so_du_doan_theo_nhom': counts,
    'kiem_dinh': tests.round(6).to_dict(orient='records'),
    'case_study': {k: case[k] for k in ('segment', 'user_id', 'chain', 'focus_names', 'accuracy_next')},
}
(OUT / 'tom_tat.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=float))
!ls -la {OUT}